# GRAM Recurrent Qwen - Clean Resume

Fresh Colab Pro+ notebook for resuming after the successful smoke tests.

Runtime settings:

- Runtime type: Python 3
- Hardware accelerator: H100 GPU if available, then A100, then L4
- High-RAM: On
- Runtime version: Latest recommended

Upload `gram-recurrent-qwen-colab-upload.zip` when prompted.

## 0. Runtime Check

In [ ]:
import os, sys, psutil, torch
from pathlib import Path

print('python', sys.version)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
print('ram_gb', psutil.virtual_memory().total / 1e9)
!nvidia-smi || true

## 1. Install Dependencies

Do not reinstall `torch`; Colab's GPU runtime already provides a CUDA build.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate pyyaml pytest sentencepiece safetensors datasets

## 2. Verify Hugging Face Hub Token

Optional but recommended. This avoids anonymous rate limits and makes model/dataset downloads more reliable. The token is not printed.

In [ ]:
import os
from getpass import getpass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_HUB_TOKEN')
    except Exception:
        token = None

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
else:
    token = getpass('Paste HF token, or press Enter to skip: ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token

if token:
    from huggingface_hub import HfApi, login
    login(token=token, add_to_git_credential=False)
    who = HfApi(token=token).whoami()
    print('HF auth OK:', who.get('name') or who.get('email') or 'authenticated user')
else:
    print('HF auth skipped; downloads will use anonymous Hub access.')

## 3. Upload Project Zip

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path
from google.colab import files

PROJECT_ROOT = Path('/content/gram-recurrent-qwen')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload gram-recurrent-qwen-colab-upload.zip'

with zipfile.ZipFile(zip_names[0]) as zf:
    zf.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT)

print('project root:', PROJECT_ROOT)
!find . -maxdepth 2 -type f | sort | head -80

## 4. Constants

In [ ]:
import torch

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
SPLIT = '6,18'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TRAIN_DTYPE = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else ('float16' if torch.cuda.is_available() else 'float32')
ADAPTER_DTYPE = 'float32'
IDENTITY_DTYPE = 'float32'
IDENTITY_ATTN = 'eager'

print({
    'MODEL_NAME': MODEL_NAME,
    'SPLIT': SPLIT,
    'DEVICE': DEVICE,
    'TRAIN_DTYPE': TRAIN_DTYPE,
    'ADAPTER_DTYPE': ADAPTER_DTYPE,
    'IDENTITY_DTYPE': IDENTITY_DTYPE,
    'IDENTITY_ATTN': IDENTITY_ATTN,
})

## 5. Unit Tests

In [ ]:
!python -m pytest -q tests

## 6. Phase 0 Strict Identity Gate

This is the real identity criterion: float32 plus eager attention. Expected result from the previous session: `max_abs_diff=0.0`.

In [ ]:
!python eval/eval_identity.py \
  --model_name {MODEL_NAME} \
  --split {SPLIT} \
  --dtype {IDENTITY_DTYPE} \
  --attn_implementation {IDENTITY_ATTN} \
  --device {DEVICE} \
  --threshold 1e-3 | tee identity_gate.log

!grep -E "max_abs_diff|mean_abs_diff|PASS|FAIL" identity_gate.log

## 7. Phase 1 Telemetry Sanity Check

Expected untrained halting weights are approximately `[0.25, 0.1875, 0.140625, 0.421875]` with mean loop depth around `2.73`.

In [ ]:
!python eval/eval_halting.py \
  --model_name {MODEL_NAME} \
  --split {SPLIT} \
  --max_loops 4 \
  --dtype {TRAIN_DTYPE} \
  --device {DEVICE}

## 8. Phase 2 Diagnostic Trajectory Check

The diagnostic scale is only to prove stochastic injection moves hidden states. Do not use these values as training defaults.

In [ ]:
!python eval/eval_trajectories.py \
  --model_name {MODEL_NAME} \
  --split {SPLIT} \
  --max_loops 4 \
  --num_trajectories 2 \
  --dtype {TRAIN_DTYPE} \
  --device {DEVICE} \
  --diagnostic_latent_scale 1.0 \
  --diagnostic_adapter_std 0.02

## 9. Prepare Opus Reasoning Dataset

Previous run produced 886 train and 46 validation examples with `--limit 1000 --max_total_tokens 2048`.

In [ ]:
!python training/prepare_hf_reasoning_jsonl.py \
  --dataset_id lordx64/reasoning-distill-opus-4-7-max-sft \
  --tokenizer_name {MODEL_NAME} \
  --output_jsonl data/opus47_train.jsonl \
  --val_jsonl data/opus47_val.jsonl \
  --limit 1000 \
  --max_total_tokens 2048

!wc -l data/opus47_train.jsonl data/opus47_val.jsonl
!head -1 data/opus47_train.jsonl

## 10. Phase 1 G4 Stability Run

The previous 200-step non-toy run produced NaNs on a G4 runtime. This is a 50-step stability pass with fp32 adapters, lower LR, shorter max length, and fail-fast NaN guards.

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path('config/qwen_0_5b_phase1.yaml').read_text())
cfg['model_name'] = MODEL_NAME
cfg['dtype'] = TRAIN_DTYPE
cfg['layer_split'] = SPLIT
cfg['adapter_dtype'] = ADAPTER_DTYPE
cfg['max_length'] = 512
cfg['max_loops'] = 4
cfg['batch_size'] = 1
cfg['max_steps'] = 50
cfg['log_every'] = 5
cfg['learning_rate'] = 1e-5
cfg['beta'] = 0.05
cfg['initial_halt_prob'] = 0.15
cfg['max_grad_norm'] = 0.3
cfg['output_dir'] = 'outputs/qwen_0_5b_phase1_g4_stability_50'

Path('config/colab_phase1_g4_stability_50.yaml').write_text(
    yaml.safe_dump(cfg, sort_keys=False),
    encoding='utf-8',
)

!cat config/colab_phase1_g4_stability_50.yaml

In [ ]:
!python training/train_phase1_ponder.py \
  --config config/colab_phase1_g4_stability_50.yaml \
  --train_jsonl data/opus47_train.jsonl \
  --device {DEVICE}

## 11. Validate Phase 1 Checkpoint

Acceptance target for `max_loops=4`: finite validation metrics, no NaNs, comfortable VRAM headroom, and mean validation loop depth closer to `2-3`, not near `1.0`. Continue to 200 steps only if this cell looks sane.

In [ ]:
PHASE1_CKPT = 'outputs/qwen_0_5b_phase1_g4_stability_50/phase1_step_50.pt'

!python eval/eval_jsonl.py \
  --model_name {MODEL_NAME} \
  --data_jsonl data/opus47_val.jsonl \
  --checkpoint {PHASE1_CKPT} \
  --split {SPLIT} \
  --max_loops 4 \
  --max_length 512 \
  --beta 0.05 \
  --dtype {TRAIN_DTYPE} \
  --adapter_dtype {ADAPTER_DTYPE} \
  --device {DEVICE}

## 12. Optional Phase 2 Smoke From Phase 1 Checkpoint

Run only after Phase 1 validation loop depth is acceptable.

In [ ]:
RUN_PHASE2_SMOKE = False

if RUN_PHASE2_SMOKE:
    cfg2 = yaml.safe_load(Path('config/qwen_0_5b_phase2.yaml').read_text())
    cfg2['model_name'] = MODEL_NAME
    cfg2['dtype'] = TRAIN_DTYPE
    cfg2['layer_split'] = SPLIT
    cfg2['adapter_dtype'] = ADAPTER_DTYPE
    cfg2['max_length'] = 512
    cfg2['max_loops'] = 4
    cfg2['num_trajectories'] = 2
    cfg2['latent_scale_init'] = 0.05
    cfg2['latent_adapter_std'] = 0.001
    cfg2['latent_injection_mode'] = 'post'
    cfg2['max_steps'] = 50
    cfg2['log_every'] = 10
    cfg2['learning_rate'] = 1e-5
    cfg2['max_grad_norm'] = 0.3
    cfg2['resume_from'] = PHASE1_CKPT
    cfg2['output_dir'] = 'outputs/qwen_0_5b_phase2_opus47_smoke_from_phase1'
    Path('config/colab_phase2_opus47_smoke.yaml').write_text(yaml.safe_dump(cfg2, sort_keys=False))
    !python training/train_phase2_stochastic.py --config config/colab_phase2_opus47_smoke.yaml --train_jsonl data/opus47_train.jsonl --device {DEVICE}
else:
    print('Phase 2 smoke disabled. Set RUN_PHASE2_SMOKE=True after Phase 1 validation looks sane.')

## 13. Save Outputs To Drive

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/gram-recurrent-qwen-colab-runs')
    target.mkdir(parents=True, exist_ok=True)
    if Path('outputs').exists():
        shutil.copytree('outputs', target / 'outputs', dirs_exist_ok=True)
    if Path('config').exists():
        shutil.copytree('config', target / 'config', dirs_exist_ok=True)
    print('saved to', target)
else:
    !find outputs -maxdepth 3 -type f 2>/dev/null || true